# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and process the FAIR² dataset (second primary colorectal cancer survivors, including MSI-H status and clinicopathological variables) using the `mlcroissant` library and pandas.

### Dataset Source
The dataset is described by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Metadata object
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date published: {metadata.date_published}")

## 2. Data Overview
Explore and list the available Record Sets (tables), their Field `@id`s, and describe their contents.

**Note:** All entities are referenced by their `@id` per Croissant best practices.

In [ ]:
# List all record sets within the data package
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = dataset._jsonld.get('recordSet', [])  # fallback for older Croissant versions

print("Available Record Sets and their field @id's:")

record_set_ids = []
fields_by_record_set = {}

for rs in dataset.record_sets:
    print(f"- Record Set @id: {rs.id}")
    record_set_ids.append(rs.id)
    field_ids = [field.id for field in rs.fields]
    fields_by_record_set[rs.id] = field_ids
    print(f"    Field @ids: {field_ids}")

Let's briefly sample a few records from each Record Set to inspect their structure. This also helps identify which sets to analyze.

In [ ]:
# Print samples (1-2 rows) from each record set using their @id
for record_set_id in record_set_ids:
    print(f"\nRecord Set: {record_set_id}")
    try:
        gen = dataset.records(record_set=record_set_id)
        for i, record in enumerate(gen):
            print(record)
            if i >= 1:
                break
    except Exception as e:
        print(f"  Could not load: {e}")

## 3. Data Extraction
Load a Record Set from the dataset into a pandas DataFrame for further analysis.

**We will use the full record set `@id`s for reference.**

In [ ]:
# For this dataset, the main patient-level record set is typically the primary record set
record_sets = record_set_ids  # previously collected

# Load all record sets into DataFrames
dataframes = {}
for rec_id in record_sets:
    try:
        records = list(dataset.records(record_set=rec_id))
        if len(records) > 0:
            dataframes[rec_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for {rec_id}, shape: {dataframes[rec_id].shape}")
        else:
            print(f"Record set {rec_id} -- No data (empty).")
    except Exception as ex:
        print(f"Failed to load {rec_id}: {ex}")

# For demonstration, pick the first non-empty record set
main_rs_id = None
for rec_id in record_sets:
    if rec_id in dataframes and not dataframes[rec_id].empty:
        main_rs_id = rec_id
        break
if main_rs_id is None:
    raise ValueError("No main record set could be loaded as a DataFrame.")

print(f"\nMain analysis record set: {main_rs_id}")
print("Fields available (@id as columns):")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We will:
* Select a numeric field by its Field `@id`;
* Filter for rows where this field's value exceeds a threshold;
* Normalize that field;
* Optionally, group and aggregate by another attribute (if available).

All operations reference columns by their `@id`.

In [ ]:
df = dataframes[main_rs_id]

# Identify possible numeric fields by inspecting column names and dtypes
print("\nInspecting columns and types:")
print(df.dtypes)

# For demonstration, we try common field IDs -- change according to the real dataset's fields
possible_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype.kind in 'fi']
print(f"Possible numeric fields: {possible_numeric_fields}")

# Choose the first detected numeric field for EDA, or prompt if not found
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    raise ValueError("No numeric fields found.")

# Attempt conversion to numeric (some columns might have strings/nan)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].median()  # or any chosen value
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records in '{main_rs_id}' with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the field (z-score)
col_norm = numeric_field_id + "_normalized"
filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, col_norm]].head())

# Attempt to group by another field (categorical)
possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'msi' in col.lower() or 'status' in col.lower() or df[col].dtype == object]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize the distribution of the key numeric variable as a histogram, and possibly compare distributions across a key categorical variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field is available, plot boxplot/grouped comparison
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    if group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We loaded the FAIR² dataset as described by its Croissant schema using `mlcroissant`.
- We explored record sets and their fields by their unique `@id` references, and loaded data into pandas DataFrames.
- Basic EDA and normalization were performed on a key numeric column, and results were grouped by a relevant categorical variable.
- Data distributions were visualized, facilitating clinical study or ML prototyping for second primary colorectal cancer.

**Next steps:** Consult dataset metadata or Croissant fields for further clinical variable analysis or tailored machine learning.